# Day 44: Nested Functions

In Python, functions are first-class objects — you can define them inside other functions,
pass them around, and return them. This opens up powerful patterns like **closures** and **factory functions**.

## 1. Functions Inside Functions

You can define a function anywhere a statement is allowed — including inside another function.
The inner function is only visible inside the outer one.

In [ ]:
def outer():
    print('outer: start')

    def inner():
        print('inner: called')

    inner()  # call the inner function
    print('outer: end')

outer()
# inner()  # NameError — inner is not visible here


## 2. Enclosing Scope — the E in LEGB

This is where the **E** in LEGB finally comes into play.
The inner function can read variables from the outer function's local scope — no keyword needed.

In [ ]:
def outer():
    message = 'Hello from outer!'  # enclosing scope for inner

    def inner():
        print(message)  # reads enclosing scope — allowed

    inner()

outer()


In [ ]:
# LEGB in action with all four scopes visible
x = 'global'

def outer():
    x = 'enclosing'

    def inner():
        # no local x — searches Enclosing before Global
        print(x)  # enclosing

    inner()

outer()
print(x)  # global


## 3. Closures

A **closure** is an inner function that is **returned** from the outer function.
Even after the outer function has returned and its local variables are gone,
the inner function still **remembers** (has captured) the values it needs.

In [ ]:
def make_greeter(greeting):     # outer
    def greet(name):            # inner — captures 'greeting'
        print(f'{greeting}, {name}!')
    return greet                # return the inner function itself

hello = make_greeter('Hello')  # make_greeter() has finished, but...
hi    = make_greeter('Hi')

hello('Alice')   # Hello, Alice!  — greeting is still remembered
hi('Bob')        # Hi, Bob!
hello('Charlie') # Hello, Charlie!


## 4. Factory Functions

A **factory function** is an outer function whose job is to create and return
a customised inner function. The argument to the factory becomes a captured variable
inside the returned function.

In [ ]:
def make_multiplier(n):
    def multiply(x):
        return x * n   # n is captured from make_multiplier
    return multiply

double = make_multiplier(2)
triple = make_multiplier(3)

print(double(5))   # 10
print(triple(5))   # 15
print(double(10))  # 20


In [ ]:
# Factory for an adder
def make_adder(n):
    def add(x):
        return x + n
    return add

add5  = make_adder(5)
add10 = make_adder(10)

print(add5(3))    # 8
print(add10(3))   # 13
print(add5(add10(1)))  # add10(1)=11, add5(11)=16


## 5. The `nonlocal` Keyword

Just as `global` lets you reassign a module-level variable from inside a function,
`nonlocal` lets you reassign an **enclosing** variable from inside a nested function.

Without it, any assignment inside the inner function creates a new local variable.

In [ ]:
def make_counter():
    count = 0          # enclosing variable

    def increment():
        nonlocal count  # I mean the enclosing count
        count += 1
        return count

    return increment

counter = make_counter()
print(counter())  # 1
print(counter())  # 2
print(counter())  # 3

# Each call to make_counter() creates an independent counter
other = make_counter()
print(other())    # 1  — completely separate count


In [ ]:
# Without nonlocal — UnboundLocalError (same logic as 'global')
def make_broken_counter():
    count = 0

    def increment():
        count += 1  # UnboundLocalError — Python treats this as a local
        return count

    return increment

# c = make_broken_counter()
# c()  # uncomment to see the error
